In [ ]:
import numpy as np
import pandas as pd
import wandb
import torch
from torch import optim
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
from tqdm import tqdm
import os
# from model import CNNModel  
from fractal_sweep_config import sweep_config

In [ ]:
# importing loacal modules
from src.activations import (relu, squared_relu, cubic_relu)
from src.base_functions import (relu_H5, relu_H5_d, relu_H5_dd,
                                square_H5, square_H5_d, square_H5_dd,
                                cubic_H5, cubic_H5_d, cubic_H5_dd)
from src.fractal_functions import (pointwise_fractal, alpha_fractalize, alpha_fractalize_first_derivative)
from src.tools import get_transforms

In [ ]:
a = -1
b = 1
n_subintervals = 6
n_iter = 2
alpha = [0, 0, 0, 0.01, 0.02, 0.03]

In [ ]:
relu_fractal = alpha_fractalize(relu, relu_H5, a, b, n_subintervals, alpha, n_iter, True)
d_relu_fractal = alpha_fractalize_first_derivative(relu, relu_H5_d, a, b, n_subintervals, alpha, n_iter, True)

squared_relu_fractal = alpha_fractalize(squared_relu, square_H5, a, b, n_subintervals, alpha, n_iter, True)
d_squared_relu_fractal = alpha_fractalize_first_derivative(squared_relu, square_H5_d, a, b, n_subintervals, alpha, n_iter, True)

cubic_relu_fractal = alpha_fractalize(cubic_relu, cubic_H5, a, b, n_subintervals, alpha, n_iter, True)
d_cubic_relu_fractal = alpha_fractalize_first_derivative(cubic_relu, cubic_H5_d, a, b, n_subintervals, alpha, n_iter, True)

In [ ]:
# Helper function to get activation function by name
def get_activation(name, input):
    name = name.lower()
    if name == 'f_relu': return pointwise_fractal(input, relu_fractal)
    if name == 'f_squared_relu': return pointwise_fractal(input, squared_relu_fractal)
    if name == 'f_cubic_relu': return pointwise_fractal(input, cubic_relu_fractal)
    raise ValueError(f"Unsupported activation: {name}")


class CNNModel(nn.Module):      
    def __init__(self, 
                 filters,                    # List of filters => Controls number of conv layers and filter sizes  
                 kernel_size,                # Size of filters  
                 activation,                 # Activation function  
                 dropout,                    # Dropout rate (optional)
                 use_batchnorm,              # Whether to use batch norm (optional)
                 input_shape=(3, 256, 256),  # Input shape compatible with iNaturalist dataset  
                 dense_units=256,            # Number of neurons in the dense (fully connected) layer  
                 num_classes=10):            # Output layer with 10 neurons  

        super().__init__()
        layers = []
        in_channels = input_shape[0]

        # Building conv-activation-maxpool blocks 
        for out_channels in filters:
            layers.append(nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, padding=1))  # Conv layer
            if use_batchnorm:
                layers.append(nn.BatchNorm2d(out_channels))   # Optional BatchNorm
            layers.append(get_activation(activation))         # Activation function 
            layers.append(nn.MaxPool2d(2))                    # Max pooling 
            if dropout > 0:
                layers.append(nn.Dropout(dropout))            # Optional dropout
            in_channels = out_channels

        # Feature extractor with 5 conv-activation-maxpool blocks 
        self.features = nn.Sequential(*layers)

        # Automatically calculate the output size after conv layers for the FC layer
        with torch.no_grad():
            dummy = torch.zeros(1, *input_shape)
            out = self.features(dummy)
            flatten_size = out.view(1, -1).shape[1]

        # Classifier block with dense + activation + dropout + output 
        self.classifier = nn.Sequential(
            nn.Linear(flatten_size, dense_units),   # First dense layer  
            get_activation(activation),             # Activation in dense layer  
            nn.Dropout(dropout),                    # Dropout
            nn.Linear(dense_units, num_classes)     # Output layer with 10 neurons  
        )

    def forward(self, x):
        x = self.features(x)            # Pass through convolutional blocks
        x = torch.flatten(x, 1)         # Flatten before fully connected layers
        return self.classifier(x)       # Output logits for classification


In [ ]:
wandb.login(key="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")

In [ ]:
# Training function
def train():
    # Initialize wandb
    wandb.init()
    config = wandb.config
    # Generating a meaningful run name using config values
    # run_name = f"run_filters-{config.filters_per_layer}_act-{config.activation}_bs-{config.batch_size}_lr-{config.learning_rate}_do-{config.dropout_rate}_bn-{config.use_batchnorm}_aug-{config.augmentation}"
    # wandb.run.name = run_name
    # wandb.run.save()

    # Transforms
    train_tf, val_tf = get_transforms(config.augmentation)

    # Loading datasets
    train_data = datasets.ImageFolder("inaturalist_12K/train", transform=train_tf)
    val_data = datasets.ImageFolder("inaturalist_12K/val", transform=val_tf)

    train_loader = DataLoader(train_data, batch_size=config.batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_data, batch_size=config.batch_size, shuffle=False, num_workers=2)

    # Preparing model
    filters = config.filters_per_layer
    model = CNNModel(
        filters=filters,
        kernel_size=3,
        activation=config.activation,
        dropout=config.dropout_rate,
        use_batchnorm=config.use_batchnorm,
        input_shape=(3, 256, 256)  
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # Loss & optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)

    # To track best validation accuracy
    best_val_acc = 0.0

    # Training loop
    for epoch in range(config.epochs):
        print(f"\nEpoch {epoch + 1}/{config.epochs}")
        print("-" * 60)
        model.train()
        total_loss, correct, total = 0, 0, 0

        for inputs, labels in tqdm(train_loader, desc="Training Progress", ncols=100, colour="magenta"):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

        train_loss = total_loss / total
        train_acc = correct / total

        # Validation loop
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for inputs, labels in tqdm(val_loader, desc="Validation Progress", ncols=100, colour="cyan"):
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                _, predicted = outputs.max(1)
                val_correct += predicted.eq(labels).sum().item()
                val_total += labels.size(0)

        val_loss /= val_total
        val_acc = val_correct / val_total

        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc*100:.2f}%")
        print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc*100:.2f}%")
        print("-" * 60)

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc
        })

    # Saving model if it's the best across all sweeps
    global_best_path = "best_accuracy.txt"
    current_best = 0.0
    
    # Readinf global best accuracy if file exists
    if os.path.exists(global_best_path):
        with open(global_best_path, "r") as f:
            try:
                current_best = float(f.read().strip())
            except:
                current_best = 0.0
    
    # Saving model only if it's better than global best
    if val_acc > current_best:
        torch.save(model.state_dict(), "best_model.pth")
        with open(global_best_path, "w") as f:
            f.write(str(val_acc))
        print(f"New global best model saved with val_acc: {val_acc:.4f}")

    wandb.finish()  
    print("Training run complete.")

In [ ]:
if __name__ == "__main__":
    sweep_id = wandb.sweep(sweep_config, project="fractal_CNN")
    wandb.agent(sweep_id, function=train, count = 2)
    print("Sweep complete")